In [1]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
from scipy.optimize import differential_evolution, curve_fit, minimize
from scipy.optimize import Bounds, LinearConstraint, NonlinearConstraint
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
from tqdm import tqdm

eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_model/"
res_DIR = "../data/results_model/"
resistance_DIR = "../data/resistance/"
# %matplotlib widget

In [2]:
def Un(sto):
    u_eq = (
        0.063
        + 0.8 * math.exp(-75 * (sto + 0.007))
        - 0.0120 * math.tanh((sto - 0.127) / 0.016)
        - 0.0118 * math.tanh((sto - 0.155) / 0.016)
        - 0.0035 * math.tanh((sto - 0.220) / 0.020)
        - 0.0095 * math.tanh((sto - 0.190) / 0.013)
        - 0.0145 * math.tanh((sto - 0.490) / 0.020)
        - 0.0800 * math.tanh((sto - 1.030) / 0.055)
    )

    return u_eq

def Up(sto):
    u_eq = (
        4.3452
        - 1.6518 * sto
        + 1.6225 * (sto ** 2)
        - 2.0843 * (sto ** 3)
        + 3.5146 * (sto ** 4)
        - 2.2166 * (sto ** 5)
        - 0.5623 * math.exp(109.451 * sto - 100.006)
    )

    return u_eq

In [3]:
cell = 1

In [4]:
cyc_no = 99
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
cyc_data_raw1['Cycle number'] = cyc_data_raw1['Cycle number'] + 1
cyc_data_raw = cyc_data_raw1[ cyc_data_raw1['Cycle number'] == cyc_no ]
cyc_data = cyc_data_raw.reset_index(drop=True)
t_c1 = cyc_data['Time [s]']-cyc_data['Time [s]'][0]
t_c1 = t_c1.values
I_c1 = cyc_data['Current [mA]']/1000
I_c1 = I_c1.values
V_c1 = cyc_data['Voltage [V]']
V_c1 = V_c1.values
E_c1 = cyc_data["Expansion [mu m]"]
E_c1 = E_c1.values


In [5]:
dfo_c = dfo_0.loc[dfo_0["N"] == cyc_no]

In [ ]:
idx_I = np.where((np.diff(I_c1)<-0.02) & (I_c1[:-1]>0) & (I_c1[:-1]<2))[0] 
# idx_I = idx_I[idx_I>50]
t = t_c1[:idx_I[0]]
V = V_c1[:idx_I[0]]
I = I_c1[:idx_I[0]]
E = E_c1[:idx_I[0]]-E_c1[0]
Qmax = max(cyc_data["Q [Ah]"])

Q = integrate.cumtrapz(I,t, initial=0)/3600 #Ah
Q = Qmax - Q
Qdata = Q
Vdata = V
fig, ax = plt.subplots(1,1,figsize=(5,4))
# ax.plot(t_c1,I_c1)
# ax.plot(t_c1[idx_I],I_c1[idx_I],'ro')
ax.plot(Q,V)

In [7]:
def OCP(X,Q):
    ocp = Up(X[3]+Q/X[2])-Un(X[1]-Q/X[0])
    return ocp

def fitfunc(x,Qdata,Vdata,R,Crate):
  model = np.concatenate([
         [OCP(x,Q)+R*Crate*5]
        for Q in Qdata
    ]
  )
  Vd = Vdata
  Vs = model
  error = Vd-Vs
  out = np.linalg.norm(error)/np.sqrt(len(Vd))
  return out

In [ ]:
cyc_no

In [ ]:
dfe_0["N"].iloc[0] = 1

In [10]:
R = dfe_0.iloc[(dfe_0['N']-cyc_no).abs().argsort()[:1]]["Rs_ave"].iloc[0]
Crate = 1/5

In [ ]:
lb = [1, 0, 1, 0]
ub = [7, 1, 7, 1]
bounds = Bounds(lb, ub)
x0 = [6,0.01,6,0.9]
result1 = minimize(fitfunc, x0, args=(Qdata,Vdata,R,0), bounds=bounds)
print(result1.x)
result2 = minimize(fitfunc, x0, args=(Qdata,Vdata,R,Crate), bounds=bounds)
print(result2.x)

In [12]:
# Cn1 = res1[0]
# print(Cn1)
# Cn2 = res2[0]
# print(Cn2)
# print((Cn1-Cn2)/Cn1)

In [ ]:
res1 = result1.x
Cn = res1[0]
Vfit1 = np.concatenate([
        [OCP(res1,Q)]
        for Q in Qdata
    ]
)
res2 = result2.x
Vfit2 = np.concatenate([
        [OCP(res2,Q)+R*Crate*5]
        for Q in Qdata
    ]
)
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(Qdata,Vdata,'k')
ax.plot(Qdata,Vfit1,'r--')
ax.plot(Qdata,Vfit2,'g-.')
ax.legend(["Data","No R","With R"])

In [14]:
def load_cycling_data_ch(cyc_data_raw1,cyc_no):
    cyc_data_raw = cyc_data_raw1[ cyc_data_raw1['Cycle number'] == cyc_no ]
    cyc_data = cyc_data_raw.reset_index(drop=True)
    t_c1 = cyc_data['Time [s]']-cyc_data['Time [s]'][0]
    t_c1 = t_c1.values
    I_c1 = cyc_data['Current [mA]']/1000
    I_c1 = I_c1.values
    V_c1 = cyc_data['Voltage [V]']
    V_c1 = V_c1.values
    E_c1 = cyc_data["Expansion [mu m]"]
    E_c1 = E_c1.values
    idx_I = np.where((np.diff(I_c1)<-0.02) & (I_c1[:-1]>0) & (I_c1[:-1]<2))[0] 
    t = t_c1[:idx_I[0]]
    V = V_c1[:idx_I[0]]
    I = I_c1[:idx_I[0]]
    E = E_c1[:idx_I[0]]-E_c1[0]
    Qmax = max(cyc_data["Q [Ah]"])
    Q1 = integrate.cumtrapz(I,t, initial=0)/3600 #Ah
    Q = Qmax - Q1
    return t,V,I,Q,E


In [15]:
lb = [1, 0, 1, 0]
ub = [7, 1, 7, 1]
bounds = Bounds(lb, ub)
x0 = [6,0.01,6,0.9]

In [ ]:
cell = 1
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
cyc_data_raw1['Cycle number'] = cyc_data_raw1['Cycle number'] + 1
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
for cyc_no in tqdm(range(1,max(cyc_data_raw1['Cycle number'])+1)):
        if cyc_no >1:
            x0 = [res[0],0.01,res[2],0.9]
        t_d,V_d,I_d,Q_d,E_d = load_cycling_data_ch(cyc_data_raw1,cyc_no)
        Q_d = Q_d[::5]
        V_d = V_d[::5]
        R = dfe_0.iloc[(dfe_0['N']-cyc_no).abs().argsort()[:1]]["Rs_ave"].iloc[0]
        Crate = 0
        # result = minimize(fitfunc, x0, args=(Q_d,V_d,R,Crate), bounds=bounds)
        # res = result.x
        # Cn = res[0]
        # Na.append(cyc_no)
        # Cna.append(Cn)
 

In [ ]:
cell = 1
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
cyc_data_raw1['Cycle number'] = cyc_data_raw1['Cycle number'] + 1
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
x0 = [6,0.9,6,0.05]
for cyc_no in tqdm(range(1,max(cyc_data_raw1['Cycle number'])+1)):
# for cyc_no in tqdm(range(1,5)):
    try:
        if cyc_no >1:
            x0 = [res[0],0.9,res[2],0.05]
        t_d,V_d,I_d,Q_d,E_d = load_cycling_data_ch(cyc_data_raw1,cyc_no)
        Q_d = Q_d[::5]
        V_d = V_d[::5]
        R = dfe_0.iloc[(dfe_0['N']-cyc_no).abs().argsort()[:1]]["Rs_ave"].iloc[0]
        Crate = 0
        result = minimize(fitfunc, x0, args=(Q_d,V_d,R,Crate), bounds=bounds)
        res = result.x
        Cn = res[0]
        Na.append(cyc_no)
        Cna.append(Cn)
    except Exception as error:
        print(cyc_no)
        print(error)

In [65]:
df1 = pd.DataFrame({"N":Na,"Cn":Cna})
# df11 = df1.copy()
# Cn_134 = (df11.query("N == 133").Cn.iloc[0] + df11.query("N == 135").Cn.iloc[0])/2
# df1 =pd.concat([df11,pd.DataFrame([[134,Cn_134]], columns=df11.columns)], ignore_index=True)
# df1 = df1.sort_values("N").reset_index(drop=True)
# df1["Type"] = "cyc"

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df1["N"],df1["Cn"])

In [18]:
# df20 = pd.read_csv("C_5.csv")

In [19]:
# fig,ax = plt.subplots(1,1,figsize=(5,4))
# ax.plot(df1["N"],df1["Cn"])
# ax.plot(df20["N"],df20["Cn"])

In [ ]:
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
x0 = [6,0.8,6,0.05]
for cyc_no in tqdm(N_0):
    # try:
        cyc_data_raw1 = dfo_0[dfo_0["N"] == cyc_no]
        Q_d = cyc_data_raw1["Q"].values
        V_d = cyc_data_raw1["V"].values
        Q_d = Q_d[::10]
        V_d = V_d[::10]
        Qmax = max(Q_d)
        Q_d = Qmax - Q_d
        if cyc_no >1:
            x0 = [res[0],0.8,res[2],0.1]
        result = minimize(fitfunc, x0, args=(Q_d,V_d,R,0), bounds=bounds)
        res = result.x
        Cn = res[0]
        Na.append(cyc_no)
        Cna.append(Cn)
    # except Exception as error:
    #     print(cyc_no)
    #     print(error)

In [73]:
df12 = pd.DataFrame({"N":Na,"Cn":Cna})
df12["Type"] = "rpt"
df_1 = pd.concat([df1,df12])
df_1 = df_1.sort_values("N").reset_index(drop=True)

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df1["N"],df1["Cn"])
ax.plot(df12["N"],df12["Cn"],'ro')
# ax.plot(df12["N"],df_1["Cn"],'ro')


In [ ]:
cell = 4
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
cyc_data_raw1['Cycle number'] = cyc_data_raw1['Cycle number'] + 1
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
for cyc_no in tqdm(range(1,max(cyc_data_raw1['Cycle number'])+1)):
    try:
        if cyc_no >1:
            x0 = [res[0],0.01,res[2],0.9]
        t_d,V_d,I_d,Q_d,E_d = load_cycling_data_ch(cyc_data_raw1,cyc_no)
        result = minimize(fitfunc, x0, args=(Q_d,V_d), bounds=bounds)
        res = result.x
        Cn = res[0]
        Na.append(cyc_no)
        Cna.append(Cn)
    except Exception as error:
        print(cyc_no)
        print(error)

In [99]:
df2 = pd.DataFrame({"N":Na,"Cn":Cna})

In [100]:
df2["Type"] = "cyc"

In [ ]:
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
for cyc_no in tqdm(N_0):
    try:
        cyc_data_raw1 = dfo_0[dfo_0["N"] == cyc_no]
        Q_d = cyc_data_raw1["Q"].values
        V_d = cyc_data_raw1["V"].values
        if cyc_no >1:
            x0 = [res[0],0.01,res[2],0.9]
        result = minimize(fitfunc, x0, args=(Q_d,V_d), bounds=bounds)
        res = result.x
        Cn = res[0]
        Na.append(cyc_no)
        Cna.append(Cn)
    except Exception as error:
        print(cyc_no)
        print(error)

In [102]:
df21 = pd.DataFrame({"N":Na,"Cn":Cna})
df21["Type"] = "rpt"
df_2 = pd.concat([df2,df21])
df_2 = df_2.sort_values("N").reset_index(drop=True)

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df2["N"],df2["Cn"])

In [ ]:
cell = 10
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
cyc_data_raw1['Cycle number'] = cyc_data_raw1['Cycle number'] + 1
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
for cyc_no in tqdm(range(1,max(cyc_data_raw1['Cycle number'])+1)):
    try:
        if cyc_no >1:
            x0 = [res[0],0.01,res[2],0.9]
        t_d,V_d,I_d,Q_d,E_d = load_cycling_data_ch(cyc_data_raw1,cyc_no)
        result = minimize(fitfunc, x0, args=(Q_d,V_d), bounds=bounds)
        res = result.x
        Cn = res[0]
        Na.append(cyc_no)
        Cna.append(Cn)
    except Exception as error:
        print(cyc_no)
        print(error)

In [23]:
df3 = pd.DataFrame({"N":Na,"Cn":Cna})
df31 = df3.copy()
Cn_358 = (df31.query("N == 357").Cn.iloc[0] + df31.query("N == 359").Cn.iloc[0])/2
df3 =pd.concat([df31,pd.DataFrame([[359,Cn_134]], columns=df31.columns)], ignore_index=True)
df3 = df3.sort_values("N").reset_index(drop=True)
df3["Type"] = "cyc"

In [ ]:
Cna = []
Na = []
x0 = [6,0.01,6,0.9]
for cyc_no in tqdm(N_0):
    try:
        cyc_data_raw1 = dfo_0[dfo_0["N"] == cyc_no]
        Q_d = cyc_data_raw1["Q"].values
        V_d = cyc_data_raw1["V"].values
        if cyc_no >1:
            x0 = [res[0],0.01,res[2],0.9]
        result = minimize(fitfunc, x0, args=(Q_d,V_d), bounds=bounds)
        res = result.x
        Cn = res[0]
        Na.append(cyc_no)
        Cna.append(Cn)
    except Exception as error:
        print(cyc_no)
        print(error)

In [25]:
df31 = pd.DataFrame({"N":Na,"Cn":Cna})
df31["Type"] = "rpt"
df_3 = pd.concat([df3,df31])
df_3 = df_3.sort_values("N").reset_index(drop=True)

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))

ax.plot(df3["N"],df3["Cn"])

In [32]:
df_1.to_csv("C_5.csv")
df_2.to_csv("C_1p5.csv")
df_3.to_csv("C_5_50.csv")

In [53]:
df_1 = df_1.sort_values(["N","Type"],ascending=True).reset_index(drop=True)
df_1 = df_1.reset_index()
df_1["N"] = df_1["index"] + 1
df_1 = df_1.drop(columns=["index"])
df_2 = df_2.sort_values(["N","Type"],ascending=True).reset_index(drop=True)
df_2 = df_2.reset_index()
df_2["N"] = df_2["index"] + 1
df_2 = df_2.drop(columns=["index"])
df_3 = df_3.sort_values(["N","Type"],ascending=True).reset_index(drop=True)
df_3 = df_3.reset_index()
df_3["N"] = df_3["index"] + 1
df_3 = df_3.drop(columns=["index"])

In [107]:
df_11 = df_1.query("Type == 'rpt'")
df_12 = df_1.query("Type == 'cyc'")
df_21 = df_2.query("Type == 'rpt'")
df_22 = df_2.query("Type == 'cyc'")
df_31 = df_3.query("Type == 'rpt'")
df_32 = df_3.query("Type == 'cyc'")

In [108]:
N1 = df_1["N"].to_numpy()
crate = 1/5
DOD = 1
A = 6.0690 + 0.0288*DOD - 0.0143*crate + 0.0138*DOD*crate
B = -0.0808 - 0.0227*DOD*crate + 0.0190*DOD + 0.0171*crate
C = 2.4634e-6 - (1.0717e-5)*DOD + (1.7256e-8)*crate + (2.5483e-6)*DOD*crate
Cn1 = A + B*N1**0.4 + C*(N1**2)*np.tanh(N1)

N2 = df_2["N"].to_numpy()
crate = 1.5
DOD = 1
A = 6.3743 - 0.1553*crate
B = -0.0083 - 0.056*crate
C = 2.8773e-05 - (2.8319e-05)*crate
Cn2 = A + B*N2**0.4 + C*(N2**2)*np.tanh(N2)

N3 = df_3["N"].to_numpy()
crate = 1/5
DOD = 0.5
A = 6.0690 + 0.0288*DOD - 0.0143*crate + 0.0138*DOD*crate
B = -0.0808 - 0.0227*DOD*crate + 0.0190*DOD + 0.0171*crate
C = 2.4634e-6 - (1.0717e-5)*DOD + (1.7256e-8)*crate + (2.5483e-6)*DOD*crate
Cn3 = A + B*N3**0.4 + C*(N3**2)*np.tanh(N3)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df_12["N"],df_12["Cn"],color="blue",linestyle="none",marker="o")
ax.plot(df_11["N"],df_11["Cn"],color="red",linestyle="none",marker="v")
ax.plot(N1,Cn1,color="green")
ax.legend(["Cycling","RPT","model"])
ax.set_title("C/5 100% DOD")
fig.savefig("C_5.png")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df_22["N"],df_22["Cn"],color="blue",linestyle="none",marker="o")
ax.plot(df_21["N"],df_21["Cn"],color="red",linestyle="none",marker="v")
ax.plot(N2,Cn2,color="green")
ax.legend(["Cycling","RPT","model"])
ax.set_title("1.5C 100% DOD")
fig.savefig("C_1p5.png")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df_32["N"],df_32["Cn"],color="blue",linestyle="none",marker="o")
ax.plot(df_31["N"],df_31["Cn"],color="red",linestyle="none",marker="v")
ax.plot(N3,Cn3,color="green")
ax.legend(["Cycling","RPT","model"])
ax.set_title("C/5 50% DOD")
fig.savefig("C_5_50.png")

In [110]:
df_1["L"] = 1
df_2["L"] = 0
df_2.loc[df_2["Type"] == 'rpt',"L"] = 1
df_3["L"] = 0
df_3.loc[df_3["Type"] == 'rpt',"L"] = 1

In [111]:
df_1.to_csv("C_5.csv")
df_2.to_csv("C_1p5.csv")
df_3.to_csv("C_5_50.csv")